# Build and evaluate a RAG pipeline

**JobReady AI lab.** Run each cell with Shift+Enter. Try the exercises before opening the solutions at the end.

**You will:**
- Chunk documents and embed them
- Compare dense, keyword and hybrid retrieval with recall@k
- Generate grounded answers with a small open model

In [ ]:
!pip -q install sentence-transformers rank_bm25 transformers
import numpy as np, re
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

## 1. A tiny knowledge base

In [ ]:
docs = {
 'refunds': 'Refund policy: you can cancel for a full refund any time before you start your first course exam. After you start an exam the payment is non-refundable.',
 'pricing': 'The exam unlock costs ₹199 in India, $3.99 in the USA and Canada, €3.49 in Europe and $2.99 elsewhere. Lessons and practice are free.',
 'camera': 'All exams use the camera. A photo is taken at the start, every 5 minutes and when you return from another tab. Photos are deleted after 30 days.',
 'referral': 'Refer & earn: you get £1 for each friend who signs up with your link, unlocks the exams and studies on 10 different days.',
 'cert': 'Pass a full 30-question exam with 70% or more to get a certificate with a verification link. Certificate IDs look like C1A2B3C4D5.'
}
chunks = list(docs.values()); names = list(docs.keys())

## 2. Retrieval: dense, BM25 and hybrid

In [ ]:
emb = SentenceTransformer('all-MiniLM-L6-v2')
C = emb.encode(chunks, normalize_embeddings=True)
bm25 = BM25Okapi([re.findall(r'\w+', c.lower()) for c in chunks])
def dense(q): return C @ emb.encode([q], normalize_embeddings=True)[0]
def keyword(q): s = bm25.get_scores(re.findall(r'\w+', q.lower())); return s / (s.max() or 1)
def hybrid(q, a=0.5): return a * dense(q) + (1 - a) * keyword(q)
gold = [('Can I get my money back after starting the test?', 'refunds'), ('How much is it in India?', 'pricing'), ('Are photos kept forever?', 'camera'), ('What do I earn for inviting friends?', 'referral'), ('What does a certificate ID look like?', 'cert'), ('price in euros', 'pricing')]
for name, f in [('dense', dense), ('bm25', keyword), ('hybrid', hybrid)]:
    hits = sum(names[int(np.argmax(f(q)))] == g for q, g in gold)
    print(f'{name:7s} recall@1 = {hits}/{len(gold)}')

## 3. Grounded generation

In [ ]:
from transformers import pipeline
gen = pipeline('text2text-generation', model='google/flan-t5-base')
def answer(q, k=2):
    top = np.argsort(-hybrid(q))[:k]
    ctx = '\n'.join(f'[{i+1}] {chunks[j]}' for i, j in enumerate(top))
    prompt = f'Answer using only the context. If it is not there, say I don\'t know.\nContext:\n{ctx}\nQuestion: {q}'
    return gen(prompt, max_new_tokens=60)[0]['generated_text'], [names[j] for j in top]
for q, _ in gold[:3]: print(q, '→', answer(q))
print(answer('Who is the CEO of Google?'))   # should say it doesn't know

## Exercises
1. Add 3 questions of your own to `gold` and re-measure recall@1.
2. Try `a=0.2` and `a=0.8` in `hybrid`. Which works best here?

In [ ]:
# your work here

## Solutions
There's no single right answer: report recall@k for each setting in a small table and pick the best on your golden set, not by intuition.